# Entendimiento y Ficha Técnica DAMA-BOK
## Dataset: Catálogo de Servicios y Solicitudes de Clientes (Landing Store)
**Proyecto**: AgroStats Intelligence Platform (`AgroStatsApp`)  
**Fase PDCO**: DEVELOPMENT | **Active Skill**: `03-development`  
**Estándares**: DAMA-DMBOK 2 (Metadata & Data Quality) | SWEBOK Cap. 2 & 3 | ISO/IEC 25010 | PEP 8  

---

### 🎯 Objetivos de la Fase:
* Caracterización de Custodio, Licencia, Frecuencia y Unidad de Observación.
* Mapeo de Preguntas de Negocio A1-J1 y supuestos operativos.
* Diccionario de datos preliminar y análisis de granularidad requerida.

### 📐 Control de Dimensionalidad y Granularidad (`docs/3-granularidad.md`):
* **Granularidad Espacial**: `Contacto / Finca Georreferenciada`
* **Granularidad Temporal**: `Registro transaccional`
* **Estrategia de Armonización**: Conversión estandarizada a código **DIVIPOLA DANE** a 5 dígitos para joins espaciales y agregación a periodicidad mensual/anual.

### 💡 Preguntas de Negocio Asociadas (`docs/1-Bateria_preguntas.md`):
* **J1: Síntesis Multicriterio**
* **Sanitización PII Ley 1581**
* **Conversión de Demanda Agroempresarial**

---


In [ ]:
# ==============================================================================
# [FASE 0: INSTALACIÓN DE DEPENDENCIAS DEL ENTORNO]
# Este bloque garantiza la instalación y disponibilidad de todas las librerías
# del proyecto en Jupyter Notebook, Google Colab o un entorno Python nuevo.
# ==============================================================================
%pip install -q pandas numpy requests python-dotenv openpyxl pypdf pyarrow pyreadstat matplotlib seaborn scipy statsmodels scikit-learn duckdb pydantic


In [ ]:
# ==============================================================================
# [FASE 0: IMPORTACIÓN Y CONFIGURACIÓN DEL ECOSISTEMA AGROSTATSAPP]
# Cumplimiento estricto de PEP 8 y carga segura de módulos propios.
# ==============================================================================
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Resolver la raíz del proyecto para importar módulos de src/
CURRENT_DIR = Path(".").resolve()
APP_ROOT = CURRENT_DIR.parent.parent
if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))

# Importación de librerías científicas y de datos estándar
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Importación de componentes del Ecosistema AgroStatsApp
from src.cleaning.sanitizer import DataSanitizer
from src.cleaning.pii_handler import PIIHandler
from src.validation.schemas import DataValidator
from src.database.db_manager import DatabaseManager
from src.modeling.sarimax_model import AgroModeler
from src.modeling.business_questions_engine import BusinessQuestionsEngine, GranularityHarmonizer

print(f"✅ Entorno verificado. Raíz de la aplicación: {APP_ROOT}")


In [ ]:
# ==============================================================================
# [CARGA DE DATOS Y CONEXIÓN AL DATA LAKEHOUSE]
# Tabla Objetivo: 'landing_leads'
# Granularidad Espacial: Contacto / Finca Georreferenciada
# Granularidad Temporal: Registro transaccional
# ==============================================================================
db = DatabaseManager(str(APP_ROOT / "data" / "processed" / "agrostats_lakehouse.db"))
parquet_file = APP_ROOT / "data/processed/landing_leads.parquet"

if parquet_file.exists():
    df = pd.read_parquet(parquet_file)
    print(f"📂 Dataset cargado desde Parquet: {parquet_file.name} | Shape: {df.shape}")
else:
    df = db.query("SELECT * FROM landing_leads")
    print(f"🗄️ Dataset cargado desde SQLite Lakehouse ('landing_leads') | Shape: {df.shape}")

# Visualizar primeras filas del dataset
df.head(5)


### 🧮 Formulación Matemática: Escenarios Paramétrico vs. No Paramétrico (`docs/4-quecomo.md`)

1. **Enfoque Paramétrico (Distribución Normal / Momentos Tradicionales):**
   $$\mu = \frac{1}{N} \sum_{i=1}^N X_i, \quad \sigma = \sqrt{\frac{1}{N-1} \sum_{i=1}^N (X_i - \mu)^2}, \quad CV = \frac{\sigma}{\mu}$$

2. **Enfoque No Paramétrico / Robusto (Resistente a Outliers y Shocks Climáticos):**
   $$\text{Mediana} = \text{Med}(X), \quad IQR = Q_3 - Q_1, \quad RSD_{IQR} = \frac{IQR}{\text{Mediana}}, \quad \hat{\beta}_{TS} = \text{Med}\left(\frac{Y_j - Y_i}{j - i}\right)$$


In [ ]:
# ==============================================================================
# [MOTOR ANALÍTICO 'QUÉ - CÓMO': ESCENARIOS PARAMÉTRICO Y NO PARAMÉTRICO]
# Resolución de preguntas de negocio de la Batería A1-J1
# ==============================================================================
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"📊 Columnas cuantitativas detectadas: {numeric_cols}")

if numeric_cols:
    target_col = numeric_cols[0]
    series_data = df[target_col].dropna()
    
    print(f"\n--- 1. ANÁLISIS PARAMÉTRICO (Catálogo de Servicios y Solicitudes de Clientes (Landing Store)) ---")
    mean_val = series_data.mean()
    std_val = series_data.std()
    cv_val = std_val / mean_val if mean_val != 0 else 0
    print(f"• Media Aritmética (μ): {mean_val:,.4f}")
    print(f"• Desviación Estándar (σ): {std_val:,.4f}")
    print(f"• Coeficiente de Variación (CV): {cv_val:.2%}")
    
    print(f"\n--- 2. ANÁLISIS NO PARAMÉTRICO / ROBUSTO ---")
    med_val = series_data.median()
    iqr_val = stats.iqr(series_data)
    mad_val = float(np.median(np.abs(series_data - med_val)))
    rsd_iqr = iqr_val / med_val if med_val != 0 else 0
    print(f"• Mediana (Med): {med_val:,.4f}")
    print(f"• Rango Intercuartílico (IQR): {iqr_val:,.4f}")
    print(f"• Desviación Absoluta de la Mediana (MAD): {mad_val:,.4f}")
    print(f"• RSD Robusto (IQR / Mediana): {rsd_iqr:.2%}")
    
    print(f"\n--- 3. CONTROL ESTADÍSTICO DE PROCESOS (REGLA 1 DE NELSON) ---")
    anomalies = AgroModeler.detect_nelson_anomalies(series_data)
    print(f"• Puntos fuera de control (|z| > 3): {anomalies.sum()} de {len(series_data)} observaciones.")
else:
    print("ℹ️ Dataset de naturaleza no estructurada / categórica. Analizando cardinalidad:")
    print(df.nunique())
